In [11]:
import os
os.environ["OMP_NUM_THREADS"] = "5"
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import KBinsDiscretizer


cp_df = pd.read_csv("datasets/cancer-pred.csv")
pd.set_option("display.max_rows", 20)
cp_df.tail(10)


,Age,Gender,BMI,Smoking,GeneticRisk,PhysicalActivity,AlcoholIntake,CancerHistory,Diagnosis
1490,44,1,16.687564,0,0,4.046537,3.678590,0,1
1491,52,1,27.967525,0,1,3.936819,1.058584,0,1
1492,66,1,19.484117,0,0,1.918732,0.726430,0,1
1493,59,1,39.266914,0,0,0.612167,1.581462,0,1
1494,79,1,17.832588,0,0,5.909161,4.880353,0,1
1495,62,1,25.090025,0,0,9.892167,1.284158,0,1
1496,31,0,33.447125,0,1,1.668297,2.280636,1,1
1497,63,1,32.613861,1,1,0.466848,0.150101,0,1
1498,55,0,25.568216,0,0,7.795317,1.986138,1,1
1499,67,1,23.663104,0,0,2.525860,2.856600,1,0


In [12]:
y = cp_df.loc[:,"Diagnosis"]
X = cp_df.loc[:,:"CancerHistory"]
X

,Age,Gender,BMI,Smoking,GeneticRisk,PhysicalActivity,AlcoholIntake,CancerHistory
0,58,1,16.085313,0,1,8.146251,4.148219,1
1,71,0,30.828784,0,1,9.361630,3.519683,0
2,48,1,38.785084,0,2,5.135179,4.728368,0
3,34,0,30.040296,0,0,9.502792,2.044636,0
4,62,1,35.479721,0,0,5.356890,3.309849,0
...,...,...,...,...,...,...,...,...
1495,62,1,25.090025,0,0,9.892167,1.284158,0
1496,31,0,33.447125,0,1,1.668297,2.280636,1
1497,63,1,32.613861,1,1,0.466848,0.150101,0
1498,55,0,25.568216,0,0,7.795317,1.986138,1


In [13]:

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=.8, random_state=42)

In [14]:
model_test = LogisticRegression(max_iter=10000)
model_test.fit(X_train, y_train)
model_test.score(X_test, y_test)

0.86

In [15]:
models = {"logreg": LogisticRegression(max_iter=10000), "svc": SVC(), "dt":DecisionTreeClassifier()}
precision_values = {"rounding": [6,5,4,3,2,1], "uniform binning": [2,3,4,5,6], "quantile binning": [2,3,4,5,6], "kmeans binning": [2,3,4,5,6]}
modes = ["both", "training_data_only", "testing_data_only"]
results = []
# huh = 0
for model_name, model in models.items():
    base_model = model.__class__(**model.get_params())
    base_model.fit(X_train, y_train)
    base_acc = base_model.score(X_test,y_test)

    results.append({"method":"base", "model": model_name, "level": 0, "mode": "base", "accuracy": base_acc})

    for method, levels in precision_values.items():
        for level in levels:
            for mode in modes:

                X_train_new = X_train.copy()
                X_test_new = X_test.copy()

                if method == "rounding":
                    if mode == "both":
                        X_train_new = X_train_new.round(level)
                        X_test_new = X_test_new.round(level)
                    elif mode == "training_data_only":
                        X_train_new = X_train_new.round(level)
                    elif mode == "testing_data_only":
                        X_test_new = X_test_new.round(level)

                elif method == "uniform binning":
                    kbd = KBinsDiscretizer(n_bins=level, encode="ordinal", strategy="uniform")
                    kbd.fit(X_train_new)
                    if mode == "both":
                        X_train_new = pd.DataFrame(kbd.transform(X_train_new), columns=X_train.columns)
                        X_test_new = pd.DataFrame(kbd.transform(X_test_new), columns=X_test.columns)
                    elif mode == "training_data_only":
                        X_train_new = pd.DataFrame(kbd.transform(X_train_new), columns=X_train.columns)
                    elif mode == "testing_data_only":
                        X_test_new = pd.DataFrame(kbd.transform(X_test_new), columns=X_test.columns)
                
                elif method == "quantile binning":
                    kbd = KBinsDiscretizer(n_bins=level, encode="ordinal", strategy="quantile")
                    kbd.fit(X_train_new)
                    if mode == "both":
                        X_train_new = pd.DataFrame(kbd.transform(X_train_new), columns=X_train.columns)
                        X_test_new = pd.DataFrame(kbd.transform(X_test_new), columns=X_test.columns)
                    elif mode == "training_data_only":
                        X_train_new = pd.DataFrame(kbd.transform(X_train_new), columns=X_train.columns)
                    elif mode == "testing_data_only":
                        X_test_new = pd.DataFrame(kbd.transform(X_test_new), columns=X_test.columns)
                
                elif method == "kmeans binning":
                    kbd = KBinsDiscretizer(n_bins=level, encode="ordinal", strategy="kmeans")
                    kbd.fit(X_train_new)
                    if mode == "both":
                        X_train_new = pd.DataFrame(kbd.transform(X_train_new), columns=X_train.columns)
                        X_test_new = pd.DataFrame(kbd.transform(X_test_new), columns=X_test.columns)
                    elif mode == "training_data_only":
                        X_train_new = pd.DataFrame(kbd.transform(X_train_new), columns=X_train.columns)
                    elif mode == "testing_data_only":
                        X_test_new = pd.DataFrame(kbd.transform(X_test_new), columns=X_test.columns)
                

                exp_model = model.__class__(**model.get_params())
                exp_model.fit(X_train_new, y_train)

                exp_acc = exp_model.score(X_test_new, y_test)
                # if method=="quantile binning" and level==2 and mode == "both":
                #     huh += 1

                
                results.append({"method": method, "model":model_name, "level": level, "mode": mode, "accuracy":exp_acc})


c:\Users\abelg\anaconda3\envs\ress\Lib\site-packages\sklearn\preprocessing\_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
c:\Users\abelg\anaconda3\envs\ress\Lib\site-packages\sklearn\preprocessing\_discretization.py:397: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 1 are removed. Consider decreasing the number of bins.
  warnings.warn(
c:\Users\abelg\anaconda3\envs\ress\Lib\site-packages\sklearn\preprocessing\_discretization.py:397: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 3 are removed. Consider decreasing the number of bins.
  warnings.warn(
c:\Users\abelg\anaconda3\envs\ress\Lib\site-packages\sklearn\preprocessing\_discretization.

In [16]:
pd.set_option("display.max_rows", None)
results = pd.DataFrame(results)
results

,method,model,level,mode,accuracy
0,base,logreg,0,base,0.860000
1,rounding,logreg,6,both,0.860000
2,rounding,logreg,6,training_data_only,0.860000
3,rounding,logreg,6,testing_data_only,0.860000
4,rounding,logreg,5,both,0.860000
5,rounding,logreg,5,training_data_only,0.860000
6,rounding,logreg,5,testing_data_only,0.860000
7,rounding,logreg,4,both,0.860000
8,rounding,logreg,4,training_data_only,0.860000
9,rounding,logreg,4,testing_data_only,0.860000


In [17]:
data = pd.Series([2.3, 5.1, 7.8, 1.2, 9.0, 4.5, 6.7, 3.3])

binned = pd.cut(data, bins=3)

print(binned)

0    (1.192, 3.8]
1      (3.8, 6.4]
2      (6.4, 9.0]
3    (1.192, 3.8]
4      (6.4, 9.0]
5      (3.8, 6.4]
6      (6.4, 9.0]
7    (1.192, 3.8]
dtype: category
Categories (3, interval[float64, right]): [(1.192, 3.8] < (3.8, 6.4] < (6.4, 9.0]]


In [18]:
X_thing = np.array([[2.3],
              [5.1],
              [7.8],
              [1.2],
              [9.0],
              [4.5]])

kbd = KBinsDiscretizer(
    n_bins=3,
    encode='ordinal',      
    strategy='uniform'     
)

X_binned = kbd.fit_transform(X_thing)

print("Binned values:")
print(X_binned)

print("Bin edges:")
print(kbd.bin_edges_)

Binned values:
[[0.]
 [1.]
 [2.]
 [0.]
 [2.]
 [1.]]
Bin edges:
[array([1.2, 3.8, 6.4, 9. ])]


In [19]:
kbd = KBinsDiscretizer(n_bins=5, encode="ordinal", strategy="uniform")

binned = kbd.fit_transform(X)

binned_df = pd.DataFrame(binned, columns=X.columns)
binned_df

,Age,Gender,BMI,Smoking,GeneticRisk,PhysicalActivity,AlcoholIntake,CancerHistory
0,3.0,4.0,0.0,0.0,2.0,4.0,4.0,4.0
1,4.0,0.0,3.0,0.0,2.0,4.0,3.0,0.0
2,2.0,4.0,4.0,0.0,4.0,2.0,4.0,0.0
3,1.0,0.0,3.0,0.0,0.0,4.0,2.0,0.0
4,3.0,4.0,4.0,0.0,0.0,2.0,3.0,0.0
5,0.0,0.0,4.0,0.0,2.0,1.0,2.0,0.0
6,4.0,4.0,1.0,0.0,0.0,4.0,3.0,0.0
7,1.0,0.0,1.0,4.0,0.0,2.0,2.0,4.0
8,3.0,4.0,3.0,0.0,2.0,2.0,0.0,4.0
9,4.0,0.0,2.0,0.0,2.0,2.0,2.0,0.0


In [20]:
kbd.bin_edges_

array([array([20., 32., 44., 56., 68., 80.]),
       array([0. , 0.2, 0.4, 0.6, 0.8, 1. ]),
       array([15.00029087, 19.99197025, 24.98364964, 29.97532902, 34.9670084 ,
              39.95868778])                                                   ,
       array([0. , 0.2, 0.4, 0.6, 0.8, 1. ]),
       array([0. , 0.4, 0.8, 1.2, 1.6, 2. ]),
       array([2.41004685e-03, 2.00084940e+00, 3.99928875e+00, 5.99772811e+00,
              7.99616746e+00, 9.99460681e+00])                               ,
       array([1.21467217e-03, 9.98394677e-01, 1.99557468e+00, 2.99275469e+00,
              3.98993469e+00, 4.98711470e+00])                               ,
       array([0. , 0.2, 0.4, 0.6, 0.8, 1. ])], dtype=object)

In [21]:
X_train_binned = kbd.fit_transform(X_train)

for i in range(X_train_binned.shape[1]):
    edges = kbd.bin_edges_[i]
    mids = (edges[:-1] + edges[1:]) / 2
    X_train_binned[:, i] = mids[X_train_binned[:, i].astype(int)]
X_train_binned

array([[50.        ,  0.1       , 27.47948933, ...,  6.99694778,
         4.48852469,  0.1       ],
       [62.        ,  0.1       , 17.49613056, ...,  4.99850843,
         4.48852469,  0.1       ],
       [62.        ,  0.9       , 37.46284809, ...,  1.00162972,
         1.49698468,  0.1       ],
       ...,
       [62.        ,  0.1       , 37.46284809, ...,  8.99538713,
         2.49416468,  0.1       ],
       [26.        ,  0.1       , 17.49613056, ...,  3.00006908,
         2.49416468,  0.1       ],
       [38.        ,  0.9       , 37.46284809, ...,  1.00162972,
         0.49980467,  0.1       ]], shape=(1200, 8))

In [22]:
df = pd.DataFrame({
    "feature1": [10, 12, 14, 16, 18],
    "feature2": [-9, 3, 5, 7, 9]
})
kbdddd = KBinsDiscretizer(n_bins=2, encode="ordinal", strategy="uniform")
newww = kbdddd.fit_transform(df)
kbdddd.bin_edges_

array([array([10., 14., 18.]), array([-9.,  0.,  9.])], dtype=object)